# Equation Comparison Analysis

## Cardiac RODEO Project - Equation Selection

This notebook provides an interactive analysis of 11 candidate equations for modeling drug-response data in cardiac organoids. Each equation captures different aspects of pharmacokinetic and pharmacodynamic behavior.

### Equations Analyzed:
1. **Polynomial** - Flexible 10-parameter polynomial surface
2. **Modified Hill** - Hill-type with concentration and time dependence
3. **Dual Exponential** - Separate benefit and toxicity exponentials
4. **Bivariate Gaussian** - 2D Gaussian peaks for localized effects
5. **Gaussian-Hill Hybrid** - Combines Hill and Gaussian components
6. **Gaussian Ridge** - Simple 2D Gaussian centered at optimal C-t
7. **PK-PD Elimination** - First-order drug elimination kinetics
8. **Adaptive Response** - Models tolerance/desensitization
9. **Recovery Model** - Reversible damage with recovery
10. **Cumulative Exposure** - AUC-dependent toxicity
11. **Biphasic Response** - Low-dose stimulation, high-dose inhibition

In [ ]:
# Cell #0
# Depends on: None
# Setup and imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Imports complete.")

In [ ]:
# Cell #1
# Depends on: 0
# Path configuration

# Find project root
current_dir = Path.cwd()
if current_dir.name == 'equation_fitting':
    PROJECT_ROOT = current_dir.parent.parent
elif (current_dir / 'Picking Equations').exists():
    PROJECT_ROOT = current_dir
else:
    PROJECT_ROOT = current_dir.parent

SCRIPT_DIR = PROJECT_ROOT / 'Picking Equations' / 'equation_fitting'
COEFF_DIR = SCRIPT_DIR / 'outputs' / 'coefficients'
CLEANED_DATA = PROJECT_ROOT / 'Cleaned_Data'

print(f"Project root: {PROJECT_ROOT}")
print(f"Coefficient dir: {COEFF_DIR}")
print(f"Coefficient dir exists: {COEFF_DIR.exists()}")

## 1. Equation Definitions

Below are the mathematical definitions of all 11 equations used in the analysis.

In [ ]:
# Cell #2
# Depends on: 0
# Equation definitions

def polynomial(C, t, R0, a1, a2, a3, a4, a5, a6, a7, a8, a9):
    """Polynomial surface"""
    return (R0 + a1*C + a2*t + a3*C**2 + a4*t**2 + a5*C*t +
            a6*C**3 + a7*t**3 + a8*C**2*t + a9*C*t**2)

def modified_hill(C, t, R0, Emax, kappa, n, m, tau):
    """Modified Hill equation"""
    tau = max(tau, 1e-9)
    exponent = -kappa * np.power(np.maximum(C, 0), n) * np.power(np.maximum(t/tau, 0), m)
    return R0 + Emax * (1 - np.exp(np.clip(exponent, -700, 0)))

def pkpd_elimination(C, t, R0, Emax, kappa, n, m, tau, k_elim):
    """PK-PD with elimination"""
    tau = max(tau, 1e-9)
    k_elim = max(k_elim, 1e-9)
    C_t = C * np.exp(-k_elim * t)  # Drug concentration over time
    exponent = -kappa * np.power(np.maximum(C_t, 0), n) * np.power(np.maximum(t/tau, 0), m)
    return R0 + Emax * (1 - np.exp(np.clip(exponent, -700, 0)))

def gaussian_ridge(C, t, R0, A, muC, muT, sigC, sigT):
    """Gaussian ridge"""
    sigC = max(sigC, 1e-6)
    sigT = max(sigT, 1e-6)
    exponent = -((C - muC)**2 / (2*sigC**2) + (t - muT)**2 / (2*sigT**2))
    return R0 + A * np.exp(np.clip(exponent, -700, 0))

def adaptive_response(C, t, R0, E_init, E_adapt, k_on, k_adapt):
    """Adaptive response"""
    initial = E_init * (1 - np.exp(-k_on * C * t))
    adaptation = E_adapt * (1 - np.exp(-k_adapt * t))
    return R0 + initial - adaptation

def cumulative_exposure(C, t, R0, Emax, kappa, n):
    """Cumulative exposure (AUC-dependent)"""
    auc = C * t
    exponent = -kappa * np.power(np.maximum(auc, 0), n)
    return R0 + Emax * (1 - np.exp(np.clip(exponent, -700, 0)))

EQUATION_FUNCTIONS = {
    'polynomial': polynomial,
    'modified_hill': modified_hill,
    'pkpd_elimination': pkpd_elimination,
    'gaussian_ridge': gaussian_ridge,
    'adaptive_response': adaptive_response,
    'cumulative_exposure': cumulative_exposure
}

print(f"Defined {len(EQUATION_FUNCTIONS)} equation functions")

## 2. Load Fitting Results

Load the R² values from the fitted coefficient files.

In [ ]:
# Cell #3
# Depends on: 1
# Load R2 results

EQUATION_NAMES = [
    'polynomial', 'modified_hill', 'dual_exponential', 'bivariate_gaussian',
    'gaussian_hill_hybrid', 'gaussian_ridge', 'pkpd_elimination',
    'adaptive_response', 'recovery_model', 'cumulative_exposure', 'biphasic_response'
]

def load_r2_results():
    """Load R2 values from coefficient CSVs."""
    results = []
    
    for eq_name in EQUATION_NAMES:
        for response_type in ['contractility', 'o2']:
            csv_path = COEFF_DIR / f"{eq_name}_coefficients_{response_type}.csv"
            if csv_path.exists():
                df = pd.read_csv(csv_path)
                if 'R2' in df.columns:
                    for _, row in df.iterrows():
                        results.append({
                            'Drug': row['Drug'],
                            'Equation': eq_name,
                            'Response': response_type.capitalize(),
                            'R2': row['R2']
                        })
    
    return pd.DataFrame(results)

df_results = load_r2_results()

if len(df_results) > 0:
    print(f"Loaded {len(df_results)} fitting results")
    print(f"\nUnique equations: {df_results['Equation'].nunique()}")
    print(f"Unique drugs: {df_results['Drug'].nunique()}")
else:
    print("No fitting results found. Run run_pipeline.py first.")
    print(f"\nLooking in: {COEFF_DIR}")

## 3. R² Summary Statistics

Compare the mean R² values across all equations and response types.

In [ ]:
# Cell #4
# Depends on: 3
# Summary statistics

if len(df_results) > 0:
    # Compute summary
    summary = df_results.groupby(['Equation', 'Response'])['R2'].agg(['mean', 'std', 'count']).reset_index()
    summary.columns = ['Equation', 'Response', 'Mean_R2', 'Std_R2', 'N']
    
    # Pivot for comparison
    pivot = summary.pivot(index='Equation', columns='Response', values='Mean_R2')
    
    # Add average
    pivot['Average'] = pivot.mean(axis=1)
    pivot = pivot.sort_values('Average', ascending=False)
    
    print("\n" + "="*60)
    print("R² SUMMARY BY EQUATION")
    print("="*60)
    print(pivot.round(4).to_string())
else:
    print("No data available for summary.")

## 4. Visualization: R² Comparison

### 4.1 Heatmap of R² Values

In [ ]:
# Cell #5
# Depends on: 4
# R2 Heatmap

if len(df_results) > 0:
    # Create heatmap data
    heatmap_data = summary.pivot(index='Equation', columns='Response', values='Mean_R2')
    
    fig, ax = plt.subplots(figsize=(8, 10))
    sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn', 
                center=0, vmin=-0.3, vmax=0.6, ax=ax,
                cbar_kws={'label': 'Mean R²'})
    
    ax.set_title('Mean R² by Equation and Response Type', fontsize=14, fontweight='bold')
    ax.set_xlabel('Response Type', fontsize=12)
    ax.set_ylabel('Equation', fontsize=12)
    
    plt.tight_layout()
    plt.show()
else:
    print("No data available for heatmap.")

### 4.2 Scatter Plot: Contractility vs O2

In [ ]:
# Cell #6
# Depends on: 4
# Scatter comparison

if len(df_results) > 0:
    pivot_scatter = summary.pivot(index='Equation', columns='Response', values='Mean_R2').reset_index()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(pivot_scatter)))
    
    for i, row in pivot_scatter.iterrows():
        c_val = row.get('Contractility', np.nan)
        o_val = row.get('O2', np.nan)
        
        if pd.notna(c_val) and pd.notna(o_val):
            ax.scatter(c_val, o_val, s=150, c=[colors[i]], 
                      edgecolors='black', linewidth=1.5, label=row['Equation'])
    
    # Diagonal line
    ax.plot([-0.5, 1], [-0.5, 1], 'k--', alpha=0.4, linewidth=2, label='Equal R²')
    
    ax.set_xlabel('Mean R² - Contractility', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean R² - O2', fontsize=12, fontweight='bold')
    ax.set_title('R² Comparison: Contractility vs O2', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='lower right', ncol=2)
    ax.set_xlim(-0.3, 0.7)
    ax.set_ylim(-0.3, 0.7)
    
    plt.tight_layout()
    plt.show()
else:
    print("No data available for scatter plot.")

### 4.3 R² Distribution by Equation

In [ ]:
# Cell #7
# Depends on: 3
# R2 distributions

if len(df_results) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Contractility
    df_c = df_results[df_results['Response'] == 'Contractility']
    for eq_name in df_c['Equation'].unique():
        eq_data = df_c[df_c['Equation'] == eq_name]['R2'].dropna()
        if len(eq_data) > 2:
            try:
                sns.kdeplot(eq_data, ax=axes[0], label=eq_name, alpha=0.7)
            except:
                pass
    
    axes[0].set_xlabel('R²', fontsize=12)
    axes[0].set_ylabel('Density', fontsize=12)
    axes[0].set_title('Contractility R² Distribution', fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=7, loc='upper left')
    axes[0].set_xlim(-0.5, 1.0)
    
    # O2
    df_o = df_results[df_results['Response'] == 'O2']
    for eq_name in df_o['Equation'].unique():
        eq_data = df_o[df_o['Equation'] == eq_name]['R2'].dropna()
        if len(eq_data) > 2:
            try:
                sns.kdeplot(eq_data, ax=axes[1], label=eq_name, alpha=0.7)
            except:
                pass
    
    axes[1].set_xlabel('R²', fontsize=12)
    axes[1].set_ylabel('Density', fontsize=12)
    axes[1].set_title('O2 R² Distribution', fontsize=13, fontweight='bold')
    axes[1].legend(fontsize=7, loc='upper left')
    axes[1].set_xlim(-0.5, 1.0)
    
    plt.tight_layout()
    plt.show()
else:
    print("No data available for distribution plots.")

## 5. Example: PK-PD Elimination Surface

Visualize the PK-PD elimination equation with typical parameters.

In [ ]:
# Cell #8
# Depends on: 2
# 3D surface example

# Create meshgrid
time = np.linspace(0, 96, 50)
dose_ratio = np.linspace(0, 2, 50)
T, D = np.meshgrid(time, dose_ratio)

# Example parameters for PK-PD elimination
R0 = 0.05       # Baseline response
Emax = 0.08     # Maximum effect
kappa = 2.0     # Effect rate constant
n = 1.5         # Concentration Hill coefficient
m = 1.2         # Time Hill coefficient
tau = 24.0      # Time constant (hours)
k_elim = 0.05   # Elimination rate constant

# Calculate response
Response = pkpd_elimination(D, T, R0, Emax, kappa, n, m, tau, k_elim)

# Create 3D plot
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(T, D, Response, cmap='viridis', alpha=0.9,
                       linewidth=0, antialiased=True)

ax.set_xlabel('Time (hours)', fontsize=11)
ax.set_ylabel('Dose Ratio (C₀/Cmax)', fontsize=11)
ax.set_zlabel('Response', fontsize=11)
ax.set_title('PK-PD Elimination Model\nExample Response Surface', fontsize=13, fontweight='bold')

ax.view_init(elev=25, azim=-158)

fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='Response')

plt.tight_layout()
plt.show()

print(f"\nParameters used:")
print(f"  R0 = {R0}, Emax = {Emax}, kappa = {kappa}")
print(f"  n = {n}, m = {m}, tau = {tau}h, k_elim = {k_elim}")

## 6. Comparing Equations: Effect of Elimination

Compare the Modified Hill equation (no elimination) with the PK-PD Elimination model.

In [ ]:
# Cell #9
# Depends on: 2, 8
# Compare with/without elimination

# Common parameters
R0 = 0.05
Emax = 0.08
kappa = 2.0
n = 1.5
m = 1.2
tau = 24.0

# Calculate surfaces
Response_hill = modified_hill(D, T, R0, Emax, kappa, n, m, tau)
Response_pkpd = pkpd_elimination(D, T, R0, Emax, kappa, n, m, tau, k_elim=0.05)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), subplot_kw={'projection': '3d'})

# Modified Hill
surf1 = axes[0].plot_surface(T, D, Response_hill, cmap='plasma', alpha=0.9)
axes[0].set_xlabel('Time (h)')
axes[0].set_ylabel('Dose Ratio')
axes[0].set_zlabel('Response')
axes[0].set_title('Modified Hill\n(No Elimination)', fontsize=12, fontweight='bold')
axes[0].view_init(elev=25, azim=-158)

# PK-PD Elimination
surf2 = axes[1].plot_surface(T, D, Response_pkpd, cmap='plasma', alpha=0.9)
axes[1].set_xlabel('Time (h)')
axes[1].set_ylabel('Dose Ratio')
axes[1].set_zlabel('Response')
axes[1].set_title('PK-PD Elimination\n(k_elim = 0.05)', fontsize=12, fontweight='bold')
axes[1].view_init(elev=25, azim=-158)

plt.tight_layout()
plt.show()

print("\nKey difference: PK-PD Elimination accounts for drug concentration decay over time,")
print("resulting in response that can plateau or decrease at late time points.")

## 7. Conclusions and Recommendations

Based on the analysis of all 11 equations:

### Key Findings:

1. **PK-PD Elimination** model provides interpretable pharmacokinetic parameters (k_elim) that relate to actual drug metabolism.

2. **Dual Exponential** and **Biphasic Response** models capture hormetic effects (low-dose stimulation, high-dose inhibition).

3. **Gaussian-based** models (Ridge, Bivariate Gaussian) excel at capturing localized optima in concentration-time space.

4. **Polynomial** offers flexibility but lacks physical interpretability.

### Recommendations:

For cardiac organoid drug screening:

- **Primary model**: PK-PD Elimination - provides drug elimination kinetics
- **Alternative**: Modified Hill - simpler, fewer parameters
- **Hormesis detection**: Biphasic Response or Dual Exponential

In [ ]:
# Cell #10
# Depends on: 4
# Final summary

if len(df_results) > 0:
    print("="*60)
    print("FINAL SUMMARY")
    print("="*60)
    
    # Best overall
    avg_by_eq = summary.groupby('Equation')['Mean_R2'].mean().sort_values(ascending=False)
    
    print("\nTop 3 Equations by Average R²:")
    for i, (eq, val) in enumerate(avg_by_eq.head(3).items()):
        print(f"  {i+1}. {eq}: R² = {val:.4f}")
    
    # Best for each response type
    print("\nBest Equation per Response Type:")
    for response in ['Contractility', 'O2']:
        df_r = summary[summary['Response'] == response]
        best = df_r.loc[df_r['Mean_R2'].idxmax()]
        print(f"  {response}: {best['Equation']} (R² = {best['Mean_R2']:.4f})")
else:
    print("Run the fitting pipeline first to generate results.")
    print("\nCommand: python run_pipeline.py")